In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import warnings, os
warnings.filterwarnings('ignore')

TRAIN_PATH  = '../data/dataset/train.csv'
TEST_PATH   = '../data/dataset/test.csv'
OUTPUT_PATH = '../submissions/submission_v10.csv'

# ─────────────────────────────────────────────────────────────
# 1. GEOHASH DECODE (no pygeohash)
# ─────────────────────────────────────────────────────────────
BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'
def decode_geohash(g):
    lat, lon = [-90., 90.], [-180., 180.]; ilon = True
    for c in g:
        b = BASE32.index(c)
        for i in range(4, -1, -1):
            bit = (b >> i) & 1
            if ilon: mid = (lon[0]+lon[1])/2; lon[bit] = mid
            else:    mid = (lat[0]+lat[1])/2; lat[bit] = mid
            ilon = not ilon
    return (lat[0]+lat[1])/2, (lon[0]+lon[1])/2

# ─────────────────────────────────────────────────────────────
# 2. LOAD
# ─────────────────────────────────────────────────────────────
print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

# ─────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
print("Feature engineering...")
geo_cache = {g: decode_geohash(g) for g in set(train_df.geohash) | set(test_df.geohash)}

for df in [train_df, test_df]:
    df['latitude']  = df['geohash'].map(lambda g: geo_cache[g][0])
    df['longitude'] = df['geohash'].map(lambda g: geo_cache[g][1])
    ts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'], df['minute'] = ts[0], ts[1]
    df['time_slot']  = df['hour'] * 4 + df['minute'] // 15
    df['slot_sin']   = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['slot_cos']   = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    df['is_weekend'] = (df['day'] % 7).isin([0, 6]).astype(int)
    df['is_rush']    = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df['is_night']   = df['hour'].isin([0, 1, 2, 3, 4, 5]).astype(int)
    df['geo3'] = df['geohash'].str[:3]
    df['geo4'] = df['geohash'].str[:4]
    df['geo5'] = df['geohash'].str[:5]
    df['road_hour']    = df['RoadType'].astype(str) + '_' + df['hour'].astype(str)
    df['geo_slot']     = df['geohash'] + '_' + df['time_slot'].astype(str)
    df['geo_hour']     = df['geohash'] + '_' + df['hour'].astype(str)
    df['road_slot']    = df['RoadType'].astype(str) + '_' + df['time_slot'].astype(str)
    df['geo5_slot']    = df['geo5'] + '_' + df['time_slot'].astype(str)
    df['lanes_x_slot'] = df['NumberofLanes'] * df['time_slot']
    df['Temperature']  = df.groupby('geohash')['Temperature'].transform(lambda x: x.fillna(x.median()))
    df['Temperature']  = df['Temperature'].fillna(df['Temperature'].median())
    for c in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
        df[c] = df[c].fillna('Unknown')

# ─────────────────────────────────────────────────────────────
# 4. LAG FEATURES (fully vectorised)
# ─────────────────────────────────────────────────────────────
print("Building lag features...")
train48 = train_df[train_df.day == 48].copy()
train49 = train_df[train_df.day == 49].copy()

d48_exact_ts   = train48.groupby(['geohash', 'timestamp'])['demand'].mean()
d48_exact_slot = train48.groupby(['geohash', 'time_slot'])['demand'].mean()
slot_med       = train_df.groupby('time_slot')['demand'].median()
geo_mean_d48   = train48.groupby('geohash')['demand'].mean()

def add_lag(df):
    d = df[['geohash', 'timestamp', 'time_slot']].copy().reset_index(drop=True)
    d = d.merge(d48_exact_ts.rename('lag').reset_index(), on=['geohash', 'timestamp'], how='left')
    for delta in [1, 2, 4, 8]:
        for sign in [1, -1]:
            sn = d['lag'].isna()
            if not sn.any(): break
            tmp = d.loc[sn, ['geohash', 'time_slot']].copy()
            tmp['adj'] = tmp['time_slot'] + sign * delta
            adf = d48_exact_slot.rename('al').reset_index()
            adf.columns = ['geohash', 'adj', 'al']
            t2  = tmp.merge(adf, on=['geohash', 'adj'], how='left')
            fi  = t2['al'].notna()
            d.loc[tmp.index[fi], 'lag'] = t2.loc[fi, 'al'].values
    sn = d['lag'].isna(); d.loc[sn, 'lag'] = d.loc[sn, 'geohash'].map(geo_mean_d48)
    sn = d['lag'].isna(); d.loc[sn, 'lag'] = d.loc[sn, 'time_slot'].map(slot_med)
    return d['lag'].values

test_df['demand_lag_d1'] = add_lag(test_df)
lag49 = add_lag(train_df[train_df.day == 49].reset_index(drop=True))
train_df['demand_lag_d1'] = np.nan
train_df.loc[train_df.day == 49, 'demand_lag_d1'] = lag49
train_df.loc[train_df.day == 48, 'demand_lag_d1'] = (train_df.loc[train_df.day == 48, 'time_slot'].map(slot_med).values)

# Derived lag signals
day_ratio = (train49.groupby('geohash')['demand'].mean() / (train48.groupby('geohash')['demand'].mean() + 1e-9)).clip(0.1, 5.0)
d49_2am   = train49[train49.timestamp == '2:0'].groupby('geohash')['demand'].mean()
d48_2am   = train48[train48.timestamp == '2:0'].groupby('geohash')['demand'].mean()
geo_2am_ratio = (d49_2am / (d48_2am + 1e-9)).clip(0.1, 10.0)
d49_agg   = train49.groupby('geohash')['demand'].agg(d49_mean='mean', d49_max='max', d49_last='last')
geo_stats = train48.groupby('geohash')['demand'].agg(geo_mean='mean', geo_max='max', geo_std='std')
geo5_slot_mean = train48.groupby(['geo5', 'time_slot'])['demand'].mean()
geo5_mean_map  = train48.groupby('geo5')['demand'].mean()

for df in [train_df, test_df]:
    df['day_ratio']     = df['geohash'].map(day_ratio).fillna(1.0)
    df['lag_adjusted']  = df['demand_lag_d1'] * df['day_ratio']
    df['geo_2am_ratio'] = df['geohash'].map(geo_2am_ratio).fillna(1.0)
    df['lag_interp']    = df['demand_lag_d1'] * df['geo_2am_ratio']
    for col in ['d49_mean', 'd49_max', 'd49_last']:
        df[col] = df['geohash'].map(d49_agg[col]).fillna(df['demand_lag_d1'])
    df['geo_mean'] = df['geohash'].map(geo_stats['geo_mean']).fillna(df['demand_lag_d1'])
    df['geo_max']  = df['geohash'].map(geo_stats['geo_max']).fillna(df['demand_lag_d1'])
    df['geo_std']  = df['geohash'].map(geo_stats['geo_std']).fillna(0)
    df['lag_norm'] = df['demand_lag_d1'] / (df['geo_max'] + 1e-9)
    g5df = geo5_slot_mean.rename('geo5_slot_demand').reset_index()
    dm   = df[['geo5', 'time_slot']].merge(g5df, on=['geo5', 'time_slot'], how='left')
    df['geo5_slot_demand'] = (dm['geo5_slot_demand'].fillna(df['geo5'].map(geo5_mean_map)).fillna(df['demand_lag_d1']).values)
    df['geo5_mean_demand'] = df['geo5'].map(geo5_mean_map).fillna(df['demand_lag_d1'])

# ─────────────────────────────────────────────────────────────
# 5. K-FOLD TARGET ENCODING
# ─────────────────────────────────────────────────────────────
print("K-Fold target encoding...")
def kfold_te(tr, te, col, folds=5):
    res = np.zeros(len(tr))
    kf  = KFold(n_splits=folds, shuffle=True, random_state=42)
    for ti, vi in kf.split(tr):
        mp = tr.iloc[ti].groupby(col)['demand'].mean().to_dict()
        res[vi] = tr[col].iloc[vi].astype(str).map(mp)
    gm = tr.groupby(col)['demand'].mean().to_dict(); om = tr['demand'].mean()
    return (pd.Series(res).fillna(om).values, pd.Series(te[col].astype(str).map(gm)).fillna(om).values)

te_cols = ['geohash', 'geo3', 'geo4', 'geo5', 'geo_slot', 'geo_hour', 'geo5_slot', 'RoadType', 'Weather', 'road_hour', 'road_slot']
for col in te_cols:
    train_df[f'{col}_te'], test_df[f'{col}_te'] = kfold_te(train_df, test_df, col)

# ─────────────────────────────────────────────────────────────
# 6. FEATURE SET COMPILATION & TARGET TRANSFORM
# ─────────────────────────────────────────────────────────────
feats = (
    ['latitude', 'longitude', 'hour', 'minute', 'time_slot', 'is_weekend', 'is_rush', 'is_night',
     'slot_sin', 'slot_cos', 'hour_sin', 'hour_cos', 'NumberofLanes', 'Temperature', 'lanes_x_slot',
     'demand_lag_d1', 'lag_adjusted', 'lag_interp', 'day_ratio', 'geo_2am_ratio', 'd49_mean', 'd49_max', 'd49_last',
     'geo_mean', 'geo_max', 'geo_std', 'lag_norm', 'geo5_slot_demand', 'geo5_mean_demand'] +
    [f'{c}_te' for c in te_cols]
)
print(f"Total features assembled: {len(feats)}")

for col in feats:
    for df in [train_df, test_df]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

X      = train_df[feats].values
y      = train_df['demand'].values
X_test = test_df[feats].values
day49_mask = train_df['day'].values == 49

# CRITICAL COMPLIANCE STRATEGY: Train on Log Space to minimize distribution skew
y_log = np.log1p(y)

# ─────────────────────────────────────────────────────────────
# 7. MODEL A — 3-SEED LIGHTGBM
# ─────────────────────────────────────────────────────────────
print("\nTraining Engine [1/4]: LightGBM (3 seeds × 5 folds)...")
lgb_params = dict(
    n_estimators=3000, learning_rate=0.02, num_leaves=127,
    max_depth=8, subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0, min_child_samples=10,
    n_jobs=-1, importance_type='gain',
)

lgb_preds = np.zeros(len(X_test))
lgb_oof   = np.zeros(len(X))

for seed in [42, 2024, 888]:
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    oof_seed = np.zeros(len(X))
    for fold, (ti, vi) in enumerate(kf.split(X, y_log)):
        m = lgb.LGBMRegressor(**lgb_params, random_state=seed, verbose=-1)
        m.fit(X[ti], y_log[ti], eval_set=[(X[vi], y_log[vi])], callbacks=[lgb.early_stopping(150, verbose=False)])
        oof_seed[vi] = m.predict(X[vi])
        lgb_preds   += m.predict(X_test) / (5 * 3)
    lgb_oof += oof_seed / 3

lgb_oof_exp = np.expm1(lgb_oof)
lgb_preds_exp = np.expm1(lgb_preds)
print(f"-> LGBM Combined OOF R2: {r2_score(y, lgb_oof_exp):.4f} | Day49: {r2_score(y[day49_mask], lgb_oof_exp[day49_mask]):.4f}")

# ─────────────────────────────────────────────────────────────
# 8. MODEL B — EXTRATREES REGRESSOR
# ─────────────────────────────────────────────────────────────
print("\nTraining Engine [2/4]: ExtraTrees (seed 2024, 5 folds)...")
et_preds = np.zeros(len(X_test))
et_oof   = np.zeros(len(X))
kf = KFold(n_splits=5, shuffle=True, random_state=2024)

for fold, (ti, vi) in enumerate(kf.split(X, y_log)):
    m = ExtraTreesRegressor(n_estimators=200, max_features=0.5, min_samples_leaf=5, n_jobs=-1, random_state=2024)
    m.fit(X[ti], y_log[ti])
    et_oof[vi]  = m.predict(X[vi])
    et_preds   += m.predict(X_test) / 5

et_oof_exp = np.expm1(et_oof)
et_preds_exp = np.expm1(et_preds)
print(f"-> ExtraTrees OOF R2: {r2_score(y, et_oof_exp):.4f} | Day49: {r2_score(y[day49_mask], et_oof_exp[day49_mask]):.4f}")

# ─────────────────────────────────────────────────────────────
# 9. MODEL C — CATBOOST REGRESSOR
# ─────────────────────────────────────────────────────────────
print("\nTraining Engine [3/4]: CatBoost (seed 42, 5 folds)...")
cb_preds = np.zeros(len(X_test))
cb_oof   = np.zeros(len(X))
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (ti, vi) in enumerate(kf.split(X, y_log)):
    m = CatBoostRegressor(iterations=2500, learning_rate=0.03, depth=7, loss_function='RMSE', verbose=0, random_seed=42)
    m.fit(X[ti], y_log[ti], eval_set=(X[vi], y_log[vi]), early_stopping_rounds=150)
    cb_oof[vi]  = m.predict(X[vi])
    cb_preds   += m.predict(X_test) / 5

cb_oof_exp = np.expm1(cb_oof)
cb_preds_exp = np.expm1(cb_preds)
print(f"-> CatBoost OOF R2: {r2_score(y, cb_oof_exp):.4f} | Day49: {r2_score(y[day49_mask], cb_oof_exp[day49_mask]):.4f}")

# ─────────────────────────────────────────────────────────────
# 10. MODEL D — XGBOOST REGRESSOR (HIST)
# ─────────────────────────────────────────────────────────────
print("\nTraining Engine [4/4]: XGBoost Hist (seed 2026, 5 folds)...")
xgb_preds = np.zeros(len(X_test))
xgb_oof   = np.zeros(len(X))

xgb_params = {
    'n_estimators': 3000, 'learning_rate': 0.02, 'max_depth': 7,
    'subsample': 0.8, 'colsample_bytree': 0.7, 'tree_method': 'hist',
    'random_state': 2026, 'n_jobs': -1, 'early_stopping_rounds': 150
}
kf = KFold(n_splits=5, shuffle=True, random_state=2026)

for fold, (ti, vi) in enumerate(kf.split(X, y_log)):
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X[ti], y_log[ti], eval_set=[(X[vi], y_log[vi])], verbose=False)
    xgb_oof[vi]  = m.predict(X[vi])
    xgb_preds   += m.predict(X_test) / 5

xgb_oof_exp = np.expm1(xgb_oof)
xgb_preds_exp = np.expm1(xgb_preds)
print(f"-> XGBoost OOF R2: {r2_score(y, xgb_oof_exp):.4f} | Day49: {r2_score(y[day49_mask], xgb_oof_exp[day49_mask]):.4f}")

# ─────────────────────────────────────────────────────────────
# 11. MATHEMATICALLY OPTIMAL META-STACKING BLEND
# ─────────────────────────────────────────────────────────────
print("\nCalculating optimal stacking weights via linear meta-model...")
stack_X = np.column_stack([lgb_oof_exp, et_oof_exp, cb_oof_exp, xgb_oof_exp])

# Stacking on the true scale target to directly optimize R2 metric performance
stacker = LinearRegression(positive=True)
stacker.fit(stack_X, y)
weights = stacker.coef_ / stacker.coef_.sum()
print(f"Optimal Quad-Weights -> LGBM: {weights[0]:.3f} | ET: {weights[1]:.3f} | CatBoost: {weights[2]:.3f} | XGBoost: {weights[3]:.3f}")

blend_oof = stacker.predict(stack_X)
stack_test = np.column_stack([lgb_preds_exp, et_preds_exp, cb_preds_exp, xgb_preds_exp])
final_preds = stacker.predict(stack_test)

print(f"\n{'═'*50}")
print(f"SYSTEM GRAND BLEND OOF R2  : {r2_score(y, blend_oof):.5f}")
print(f"SYSTEM DAY 49 VALIDATION R2: {r2_score(y[day49_mask], blend_oof[day49_mask]):.5f}")
print(f"COMPETITION SCORE TRACKER  : {100*r2_score(y, blend_oof):.2f}")
print(f"{'═'*50}")

# ─────────────────────────────────────────────────────────────
# 12. SUBMISSION GENERATION
# ─────────────────────────────────────────────────────────────
final_preds = np.clip(final_preds, 0, None)
sub = pd.DataFrame({'Index': test_df['Index'], 'demand': final_preds})
sub.sort_values('Index').reset_index(drop=True).to_csv(OUTPUT_PATH, index=False)
print(f"Saved completed finalsubmission script → {OUTPUT_PATH}  Shape: {sub.shape}")

Loading data...
Feature engineering...
Building lag features...
K-Fold target encoding...
Total features assembled: 40

Training Engine [1/4]: LightGBM (3 seeds × 5 folds)...
-> LGBM Combined OOF R2: 0.9672 | Day49: 0.9612

Training Engine [2/4]: ExtraTrees (seed 2024, 5 folds)...
-> ExtraTrees OOF R2: 0.9641 | Day49: 0.9518

Training Engine [3/4]: CatBoost (seed 42, 5 folds)...
-> CatBoost OOF R2: 0.9642 | Day49: 0.9597

Training Engine [4/4]: XGBoost Hist (seed 2026, 5 folds)...
-> XGBoost OOF R2: 0.9668 | Day49: 0.9611

Calculating optimal stacking weights via linear meta-model...
Optimal Quad-Weights -> LGBM: 0.449 | ET: 0.140 | CatBoost: 0.059 | XGBoost: 0.352

══════════════════════════════════════════════════
SYSTEM GRAND BLEND OOF R2  : 0.96774
SYSTEM DAY 49 VALIDATION R2: 0.96197
COMPETITION SCORE TRACKER  : 96.77
══════════════════════════════════════════════════
Saved completed finalsubmission script → ../submissions/submission_v10.csv  Shape: (41778, 2)
